<h1>Phishing Detecter Modeling Notebook</h1>
<h2>By: Ana Madrigal, Katherine Vykhovanets, and Sydnie Dailey</h2>

<h3>The Task</h3>
This notebook will examine 102,144 urls in the final_curated_dataset(final_curated_dataset).csv dataset and use them to test and train two models: Logistic Regression and Random Classifiers Forest Classifiers. For each model, the data will be split and used in three ways: random, temporal, and domain-grouped split and the precision, recall, F1, ROC-AUC, PR-AUC, and recall at 1% and 0.1% FPR for each will be recorded.

<h3>The Performance:</h3>
The percentage of false positives and correctly classified data will measure the performance of our phishing detector.

<h3>The Data:</h3>
<table style="width:80%">
<tr>
<th>Type</th>
<th>Amount</th>
</tr>
<tr>
<td>Phishing</td>
<td>51,702</td>
</tr>
<tr>
<td>Legitimate</td>
<td>51,702</td>
</tr>
<tr>
<td>Total</td>
<td>102,144</td>
</tr>
<table>



In [1]:
#---IMPORTING NECESSARY PACKAGES DATAFRAMES AND CREATING INTIAL NECESSARY ATTRIBUTES---

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score, classification_report, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import math 
import numpy as np
from scipy.sparse import hstack
from urllib.parse import urlparse
import tldextract

R_PATH = r"ProcessedData\final_curated_dataset(final_curated_dataset).csv"
df = pd.read_csv(R_PATH)
lexical_df = pd.read_csv(R_PATH, usecols=['model_clean_url', 'label', 'timestamp'])
    #creating a dataframes: df will hold all data from urls csv and the path, query, and hostname. lexical-df is created to hold all 10 of the lexical features later

keywords_path = r"ProcessedData\Evilginx Keywords(Sheet1).csv"
keywords = pd.read_csv(keywords_path, usecols=["Evilginx Hostnames"])["Evilginx Hostnames"].dropna().astype(str).tolist()
    #importing array of keywords from github


def parse_url(url):
    """Separates cleaned urls without a scheme and returns a Series with the hostname, path, and query parameters"""
    parsed = urlparse("http://" + url)
    return({
        "hostname": parsed.hostname if parsed.hostname is not None else "",
        "path" : parsed.path if parsed.path is not None else "",
        "query" : parsed.query if parsed.query is not None else ""
    })

df[["hostname", "path", "query"]] = df["model_clean_url"].apply(lambda u: pd.Series(parse_url(u)))
    #adding attributes to df

print(df.head().to_string())
print(lexical_df.head().to_string())
print("Number of URLs: ", len(df["model_clean_url"]))
print("Number of Phishing: ", len(df[df["label"] == "phishing"]))
print("Number of Legitimate: ", len(df[df["label"] == "legitimate"]))
    #print dataframe check

                                                                                                                                           url                                                                                                                       model_clean_url       label             source             timestamp path_depth_bucket                                          hostname                                                                 path             query
0                                                                                                                         mylife.com/brickbill                                                                                                                  mylife.com/brickbill  legitimate   faizann24-benign                   NaN                 1                                        mylife.com                                                           /brickbill                  
1                             

<h1>Lexical Feautures Used:</h1>
<ol>
<li>Url length </li>
<li>Hostname Length</li>
<li>Frequency of ".", "-", "@", and "\" in the hostname</li>
<li>Number of Query Parameters</li>
<li>TLD type</li>
<li>Use of Suspicious Keywords</li>
<li>Ratio of Encoded Characters</li>
<li>Entropy of the URL</li>
<li>Number of subdirectories</li>
<li>Number of Digits in the URL</li>
</ol>


In [ ]:
#---THE 10 HAND-ENGINEERED LEXICAL FEATURES---

lexical_df["url_cleaned_length"] = df["model_clean_url"].apply(len)

lexical_df["hostname_length"] = df["hostname"].apply(len)

lexical_df["symbol_count"] = df["hostname"].str.count("/.-@")

lexical_df["num_of_parameters"] = df["query"].str.count("=")

s_tlds = ["li", "es", "sbs", "dev", "cdf", "top", "xyz"]
lexical_df["suspicious_tld"] = df["hostname"].apply(lambda tld: tld.split(".")[-1] in s_tlds).astype(int)

lexical_df["flagged_keyword"] = df["model_clean_url"].apply(lambda u: any(k in u.lower() for k in keywords)).astype(int)

lexical_df["ratio_of_encoded"] = df["model_clean_url"].str.count("%") / lexical_df["url_cleaned_length"]

def entropy(s):
    prob = [s.count(c)/len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in prob)
lexical_df["entropy"] = df["model_clean_url"].apply(entropy)

lexical_df["num_of_subdirectories"] = df["path"].str.count("/")

lexical_df["num_digits_in_url"] = df["model_clean_url"].str.count(r"\d")

print(lexical_df.head().to_string())
    #print lexical features check

                                                                                                                        model_clean_url       label             timestamp  url_cleaned_length  hostname_length  symbol_count  num_of_parameters  suspicious_tld  flagged_keyword  ratio_of_encoded   entropy  num_of_subdirectories  num_digits_in_url
0                                                                                                                  mylife.com/brickbill  legitimate                   NaN                  20               10             0                  0               0                0               0.0  3.546439                      1                  0
1                                                                                                                    wcos.wordpress.com  legitimate                   NaN                  18               18             0                  0               0                0               0.0  3.197160              

<h3>Character n-gram TF-IDF & Lexical feature values</h3>

The n-gram will analyze characters that have an n-gram greater than or equal to 3 and less than or equal to 4, convert all hcaracters to lowercase and, only analyze 200,000 features max. Since the dataset is so large it took a few tries and to get the best n-gram for the dataset. To make this process faster, I lessened the n-gram range from (3,5) to (3,4) and I decreased the amount of feautures. 


In [57]:
#CHARACTER N-GRAM TF-IDF

n_gram_tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3,4), lowercase=True, max_features=200000)

X_char = n_gram_tfidf.fit_transform(df["model_clean_url"])

X_lexical = lexical_df[[
    "url_cleaned_length", "hostname_length", "symbol_count", 
    "num_of_parameters", "suspicious_tld", "flagged_keyword", "ratio_of_encoded", "entropy",
    "num_of_subdirectories", "num_digits_in_url"]].values

X = hstack([X_char, X_lexical])
y = lexical_df["label"].values


In [ ]:
#fuction to return metrics for each model and split
def evaluate_model(model, X_test, y_test, model_type, split_type):
    """
    Evaluate any binary classifier on phishing vs legitimate URLs.
    Returns a dictionary of all required metrics.
    """

    # Convert labels to integers: phishing=1, legitimate=0
    y_test_int = (y_test == "phishing").astype(int)

    # Predictions
    y_pred = model.predict(X_test)
    y_pred_int = (y_pred == "phishing").astype(int)

    # Probabilities for ROC/PR curves
    y_prob = model.predict_proba(X_test)[:, 1]

    # Basic metrics
    precision = precision_score(y_test_int, y_pred_int)
    recall = recall_score(y_test_int, y_pred_int)
    f1 = f1_score(y_test_int, y_pred_int)

    # AUC metrics
    roc_auc = roc_auc_score(y_test_int, y_prob)
    pr_auc = average_precision_score(y_test_int, y_prob)

    # Recall at low false-positive rates
    fpr, tpr, thresholds = roc_curve(y_test_int, y_prob)
    recall_at_1 = tpr[fpr <= 0.01][-1] if np.any(fpr <= 0.01) else 0.0
    recall_at_01 = tpr[fpr <= 0.001][-1] if np.any(fpr <= 0.001) else 0.0

    return {
        "model" : model_type,
        "split" : split_type,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "recall_at_1%_FPR": recall_at_1,
        "recall_at_0.1%_FPR": recall_at_01
    }

<h3>Random Stratified Split</h3>

In [59]:
#TRANING DATA FOR RANDOM SPLIT MODELS
rX_train_full, rX_test, ry_train_full, ry_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rX_train, rX_val, ry_train, ry_val = train_test_split(rX_train_full, ry_train_full, test_size=0.2, random_state=42, stratify=ry_train_full)
    #getting validation slices from training data



Tuning

In [60]:
for C in [0.01, 0.1, 1, 10]:
    lrt = LogisticRegression(C=C, solver="saga")
    lrt.fit(rX_train, ry_train)
    score = lrt.score(rX_val, ry_val)
    print(C, score)

c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


0.01 0.7296701951906015


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


0.1 0.7297313834669278


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


1 0.729792571743254
10 0.7298537600195802


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Results

In [61]:
#RANDOM SPLIT LOGISTIC REGRESSION
lrR = LogisticRegression(max_iter=500, solver="saga", C=1.0, warm_start=True)
lrR.fit(rX_train_full, ry_train_full)

lr_random_metrics = evaluate_model(lrR, rX_test, ry_test, "Logistic Regression", "Random")
print(lr_random_metrics)

#making process correct and faster

{'model': 'Logistic Regression', 'split': 'Random', 'precision': 0.7920813737285355, 'recall': 0.7090268259252007, 'f1': 0.7482564446970088, 'roc_auc': 0.8336584080606495, 'pr_auc': 0.8530977185730481, 'recall_at_1%_FPR': np.float64(0.2924417466222831), 'recall_at_0.1%_FPR': np.float64(0.11210103779126689)}


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Tuning 

In [62]:
#RANDOM SPLIT RANDOM FOREST Tuning
rfR = RandomForestClassifier(n_estimators=75, max_depth=40, random_state=42, max_features="sqrt")

rfR.fit(rX_train, ry_train)

ry_val_pred = rfR.predict(rX_val)
print(classification_report(ry_val, ry_val_pred))

              precision    recall  f1-score   support

  legitimate       0.85      1.00      0.92      8171
    phishing       1.00      0.83      0.91      8172

    accuracy                           0.91     16343
   macro avg       0.93      0.91      0.91     16343
weighted avg       0.93      0.91      0.91     16343



In [63]:
rfR.fit(rX_train_full, ry_train_full)

rf_random_metrics = evaluate_model(rfR, rX_test, ry_test, "Random Forest", "Random")
print(rf_random_metrics)


{'model': 'Random Forest', 'split': 'Random', 'precision': 0.9960037611659615, 'recall': 0.8296455844918739, 'f1': 0.9052451661147314, 'roc_auc': 0.9809614820424895, 'pr_auc': 0.9848643513255556, 'recall_at_1%_FPR': np.float64(0.8845701977677697), 'recall_at_0.1%_FPR': np.float64(0.694634814959859)}


In [64]:
#TEMPORAL SPLIT 

#getting the phishing urls since they are the only ones with time stamps
p_df = df[["model_clean_url", "timestamp", "label"]].copy()
phish = p_df[p_df["label"] == "phishing"]       
legit = p_df[p_df["label"] == "legitimate"]     
p_time_sorted = phish.sort_values("timestamp")

#print(p_time_sorted.head().to_string())

cutoff = int(len(p_time_sorted) * 0.8)

phish_train = p_time_sorted.iloc[:cutoff]
phish_test = p_time_sorted.iloc[cutoff:]

legit_train, legit_test = train_test_split(legit, test_size=len(phish_test), random_state=42)

train_df = pd.concat([phish_train, legit_train], ignore_index=True)
test_df = pd.concat([phish_test, legit_test], ignore_index=True)

tX_train = hstack([X_char[train_df.index], X_lexical[train_df.index]])
ty_train = train_df["label"].values

tX_test = hstack([X_char[test_df.index], X_lexical[test_df.index]])
ty_test = test_df["label"].values

In [65]:
#TEMPORAL SPLIT Random Forest

rfT = RandomForestClassifier(n_estimators=75, max_depth=40, random_state=42, max_features="sqrt")
rfT.fit(tX_train, ty_train)

rf_temporal_metrics = evaluate_model(rfT, tX_test, ty_test, "Random Forest", "Temporal")
print(rf_temporal_metrics)

{'model': 'Random Forest', 'split': 'Temporal', 'precision': 0.5035460992907801, 'recall': 0.6533529123837494, 'f1': 0.5687502663087477, 'roc_auc': 0.508717938765873, 'pr_auc': 0.5083855578886423, 'recall_at_1%_FPR': np.float64(0.010181106216348507), 'recall_at_0.1%_FPR': np.float64(0.001860009789525208)}


In [66]:
#TEMPORAL SPLIT Logic Regression
lrT = LogisticRegression(max_iter=500, solver="saga", C=1.0, warm_start=True)
lrT.fit(tX_train, ty_train)

lr_temporal_metrics = evaluate_model(lrT, tX_test, ty_test, "Logistic Regression", "Temporal")
print(lr_temporal_metrics)

{'model': 'Logistic Regression', 'split': 'Temporal', 'precision': 0.5024044146629878, 'recall': 0.6238864415075869, 'f1': 0.5565938864628821, 'roc_auc': 0.5028135421286204, 'pr_auc': 0.5036305966543682, 'recall_at_1%_FPR': np.float64(0.011551639745472344), 'recall_at_0.1%_FPR': np.float64(0.0013705335291238376)}


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [67]:
#domain grouped split

df["registered_domain"] = df["model_clean_url"].apply(lambda u: (lambda ext: ext.domain + "." + ext.suffix if ext.suffix else ext.domain))

domains = df["registered_domain"].unique()

train_domains, test_domains = train_test_split(domains, test_size=0.2, random_state=42)
train_dom_df = df[df["registered_domain"].isin(train_domains)]
test_dom_df = df[df["registered_domain"].isin(test_domains)]

dX_train = hstack([X_char[train_dom_df.index], X_lexical[train_dom_df.index]])
dy_train = train_dom_df["label"].values

dX_test = hstack([X_char[test_dom_df.index], X_lexical[test_dom_df.index]])
dy_test = test_dom_df["label"].values


In [68]:
rfD = RandomForestClassifier(n_estimators=75, max_depth=40, random_state=42, max_features="sqrt")
rfD.fit(dX_train, dy_train)

rf_domain_metrics = evaluate_model(rfD, dX_test, dy_test, "Random Forest", "Domain")
print(rf_domain_metrics)

{'model': 'Random Forest', 'split': 'Domain', 'precision': 0.9970223916150548, 'recall': 0.8202038016852832, 'f1': 0.9000107515320933, 'roc_auc': 0.9835467621904163, 'pr_auc': 0.9867372696875673, 'recall_at_1%_FPR': np.float64(0.893787967862042), 'recall_at_0.1%_FPR': np.float64(0.6072898295120517)}


In [69]:
lrD = LogisticRegression(max_iter=500, solver="saga", C=1.0, warm_start=True)
lrD.fit(dX_train, dy_train)

lr_domain_metrics = evaluate_model(lrD, dX_test, dy_test, "Logistic Regression", "Domain")
print(lr_domain_metrics)

{'model': 'Logistic Regression', 'split': 'Domain', 'precision': 0.7929110600590745, 'recall': 0.7101704879482658, 'f1': 0.7492634516979376, 'roc_auc': 0.8358128529021324, 'pr_auc': 0.8523087869184676, 'recall_at_1%_FPR': np.float64(0.2962962962962963), 'recall_at_0.1%_FPR': np.float64(0.10190084264158339)}


c:\Users\sydni\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [70]:
data = [
    lr_random_metrics, rf_random_metrics, 
    lr_temporal_metrics, rf_temporal_metrics, 
    lr_domain_metrics, rf_domain_metrics
    ]

all_metrics = pd.DataFrame(data)

print(all_metrics.to_string())
print()
print(all_metrics[all_metrics["model"] == "Random Forest"].to_string())

                 model     split  precision    recall        f1   roc_auc    pr_auc  recall_at_1%_FPR  recall_at_0.1%_FPR
0  Logistic Regression    Random   0.792081  0.709027  0.748256  0.833658  0.853098          0.292442            0.112101
1        Random Forest    Random   0.996004  0.829646  0.905245  0.980961  0.984864          0.884570            0.694635
2  Logistic Regression  Temporal   0.502404  0.623886  0.556594  0.502814  0.503631          0.011552            0.001371
3        Random Forest  Temporal   0.503546  0.653353  0.568750  0.508718  0.508386          0.010181            0.001860
4  Logistic Regression    Domain   0.792911  0.710170  0.749263  0.835813  0.852309          0.296296            0.101901
5        Random Forest    Domain   0.997022  0.820204  0.900011  0.983547  0.986737          0.893788            0.607290

           model     split  precision    recall        f1   roc_auc    pr_auc  recall_at_1%_FPR  recall_at_0.1%_FPR
1  Random Forest    Random   

<h1>Interperting the Results & Explaining the Gaps</h1>

<h3>Why the Logistic Regression Models Did Poorly and the Random Forest Model was Chosen:</h3>
The logistic regression model is linear and weaker compared to the random forest model. Random forest also handles character level TF-IDF better than the logistic regression model. Additionally the random forest model is more robust, leading to better differientation between phishing and legitimate urls and higher metrics overall. Even in the random and domain splits, where logistic regression got high precision, recall, and f1 scores, the logistic regression model still doesn't hold up at ROC_AUC, PR_AUC,Recall at 1% FPR, and recall at 0.1% FPR unlike the Random Forest models with the same splits.This is because logistic regression models can't capture nonlinear patterns in each url unlik random forest.

<h3>Why the Models With A Temporal split Did Poorly:</h3>
The Temporal split has a precision of  0.503546, recall of 0.653353, PR-AUC of 0.508386, and a ROC-AUC of 0.508718. The temporal split yields worse results than the other splits because newer phishing methods are created often and they evolve over time. The model would have a difficult time classifying the test URLs since the training URLs are older and likely different which leads to a low ROC_AUC and PR_AUC. Additionally, this model as basically zero recall at 1% and 0.1% FPR so it is not good enough to be implemented.

<h3>Why the Random Forest Models with the Random and Domain split Are Most Successfull:</h3>
The random split lead to a better representation of the data than in the temporal split because that data reflected past and present phishing URL structures. This is why the random slit has a high precision of  0.996004, recall of 0.829646 and a high PR-AUC of 0.984864. The domain split led to similar results as the random split because the dataset contains a lot of URLs with the same path and very similar domains. Therefore, if these types of URLs are split amongst the testing and training data, then the model will be more precise. This is why the domain split produces a high PR-AUC of 0.986737. 

<h2>Overall Results:</h2>
The most successful performance came from the Random forest model with the domain and random splits because the training data for both reflected more patterns in the testing data. The temporal split had a subpar performance because the data was gathered at different times and subsequently led to the patterns in the newer URLs being unfamiliar to the model. 



In [75]:
#BASE RATE CHECK from RANDOM FOREST DOMAIN SPLIT

#Base-rate check: assume a mail gateway sees 1 phishing link per 10,000. Using your recall and FPR at 
#your chosen threshold, compute alarms per day for 5 million links and P(phishing | alarm). 

br_recall = 0.884570
br_FPR = 0.1
br_phish = (5000000 / 10000) 
br_TP = br_phish * br_recall
br_FP = (5000000 - br_phish) * br_FPR
br_alarms_per_day = br_FP + br_TP
br_p = br_TP / (br_TP + br_FP)

print("FPR: 1%")
print("The alarms per day: ", br_alarms_per_day)
print("P (Phishing | Alarm): ", br_p)
print()

br_recall2 = 0.6494635
br_FPR2 = 0.01
tp = br_phish * br_recall2
fp = (5000000 - br_phish) * br_FPR2
alarms_per_day = fp + tp
p = tp / alarms_per_day

print("FPR: 0.1%")
print("The alarms per day: ", alarms_per_day)
print("P (Phishing | Alarm): ", p)


FPR: 1%
The alarms per day:  500392.285
P (Phishing | Alarm):  0.0008838765369853774

FPR: 0.1%
The alarms per day:  50319.73175
P (Phishing | Alarm):  0.006453368066692843


<h2>Base Rate Check Interpertation:</h2>
These results indicate that that this detector cannot realistically be deployed as is. The alarms per day of for the FPR to be 1% and 0.1% respectively are 500392.285 and 50319.73175 which are both too many false positive URLs for a team to be looking through. Both P values show that the team would spend too much time looking for only a handful of true alerts which would waste too much time and money. 

In [72]:
#error analysis

ty_pred = lrR.predict(tX_test)

fn_indices = np.where((ty_test == "phishing") & (ty_pred == "legitimate"))[0]
false_negatives = test_df.iloc[fn_indices].head(5)

fp_indices = np.where((ty_pred == "phishing") & (ty_test == "legitimate"))[0]
false_positives = test_df.iloc[fp_indices].head(5)

print(false_negatives.to_string())
print(false_positives.to_string())


                                                                                                                                                                              model_clean_url             timestamp     label
0  www.roblox.com.pt/games/113780042580709/TRADING-Beach-Escape-1-Speed?game_id=113780042580709&game_name=TRADING-Beach-Escape-1-Speed&privateServerLinkCode=99382535900423182128231632453729  2026-08-23T12:00:05Z  phishing
1                                                                                                                                                                                   15881.xyz  2026-08-23T12:00:05Z  phishing
2                                                                                                                                                                                   68105.xyz  2026-08-23T12:00:05Z  phishing
4                                                                                                               

<h1>Observed Patterns:</h1>

<h2>False Negatives</h2>
<ol>
<li>While this url seems abnormal with some features such as its length and amount of parameters, it may score normally with the rest of the lexical features and appeared legitmate.</li>
<li>Other than the suspicious TLD and that number of digits the, this url would score normally on lexical features and the charater n-gram could have not spotted any patterns due to the short length of the url.</li>
<li>Similar to the previous url, this url ranks normally in its lexical features and it may have been too short for the character n-gram to classify any patterns</li>
<li>This url has a lot of dots however it could seem normal due to scoring normally in most lexical features  like a non-suspicious TLD, short length, and small amount of subdirectories</li>
<li>The url has a lot of parameters, dots, digits, and subdirectories however it can score normally for other lexical features and the charchter n gram representation may not have classified patterns for certain brands</li>
</ol>


<h2>False Positives</h2>
<ol>
<li>The length of this URL, the amount of dots, and the two subdirectories is similar to some of the phishing URLs, and the character n-gram could have identified its character sequences incorrectly.</li>
<li>This URL has a lot of digits and dashes that could make it resemble a phishing URL and it is also a similar length to the other URLs with a false positive.</li>
<li>This URL has some abnormal features such as many digits, more than one subdirectory, and a small hostname and the digits may have caused the character n-gram to classify this URL incorrectly.</li>
<li>The URL is short and has no digits nor subdirectories, so it may not have triggered many lexical features. However, the n-gram may not be able to classify the patterns correctly.</li>
<li>For this URL, the n-gram could have interpreted the lone parameter as a malicious pattern, and the length of the URL could be like those classified as phishing URLS</li>
</ol>


<h2>Overall & Grouping:</h2>
Patterns observed in the false negatives include the URLs containing numbers and having a large variety of lengths. All the URLs also contain some numbers which should be an indicator that they are phishing however, the lexical features could have been misinterpreted, and many legitimate URLs could contain a similar number of numbers. Additionally, the lengths of the URLs vary a lot, so maybe a lot of the training phishing URLs had a smaller range of their lengths.

The patterns observed in the false positive group are their length, the low entropy, and unusual tokens, Unlike the false negative group, this group of URLs all have a similar length that could have been innocently marked as a trait of phishing URLs. Also, these URLs don’t seem random, nor do they have a lot of digits so they would have been marked as having low entropy. Lastly, each URL has different tokens that may have been handled as if they were phishing by the character n-gram.
